# LLM SFT with QLoRA (PEFT)

Performing Supervised Fine-Tuning (SFT) for *skill acquisition* on `Qwen/Qwen2-1.5B-Instruct` while keeping GPU memory usage low by using Parameter-Efficient Fine-Tuning (PEFT).

**Objective:**
- Use a training sample size > 2000 examples (this notebook uses 2,500 from `flytech/python-codes-25k`).
- Convert the dataset to Qwen-style ChatML messages and tokenize it.
- Fine-tune the model using LoRA adapters in a quantized setup (QLoRA) and track training (e.g., with Weights & Biases).
- Save the trained adapter and show a few before/after generations to demonstrate the learned coding skill.

**QLoRA (Quantized Low-Rank Adaptation)** combines 4-bit quantization with LoRA:
- The base model weights are loaded in 4-bit (NF4) via `BitsAndBytesConfig` and kept *frozen*.
- Trainable LoRA adapter matrices are inserted into selected linear layers, so only a small number of parameters are updated.
- Much lower VRAM and compute than full fine-tuning, while often achieving similar task performance for many SFT workloads.

In practice, the small LoRA adapter weights are shared/deployed, not a full fine-tuned copy of the model.

## Setup

In [18]:
!pip install -q -U transformers peft trl bitsandbytes datasets wandb 

In [ ]:
import wandb
from huggingface_hub import login

wandb_key="" # key removed
hf_token="" # token removed

wandb.login(key=wandb_key)
login(token=hf_token)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


### Step 1: Dataset Initialization

This section pulls the `flytech/python-codes-25k` dataset required for the skill acquisition phase. To keep training times practical and VRAM usage within the limits of a single T4 GPU, data is deterministically shuffled and a subset of 2500 examples is used. (Required sample size: > 2000)

In [20]:
from datasets import load_dataset

dataset_id = "flytech/python-codes-25k"
dataset = load_dataset(dataset_id, split="train")

dataset = dataset.shuffle(seed=42).select(range(2500))

print(f"Dataset size: {len(dataset)}")
print(dataset[0].keys())

Dataset size: 2500
dict_keys(['output', 'instruction', 'input', 'text'])


### Step 2: Tokenization and ChatML Formatting

Large Language Models require text to be converted into numerical tokens using their specific dictionary. This section loads the Qwen tokenizer and restructures the raw dataset columns into a standardized format.

#### ChatML Architecture
Chat Markup Language (ChatML) is a templating structure that defines conversational boundaries using special tokens. It teaches the model how to distinguish between system instructions, the user's prompt, and its own generated code, which is important for fine-tuning.

**1. The Raw Dataset (Before Mapping)**
The original dataset provides data separated into distinct columns:
* `instruction`: The user's request (e.g., "Write a Python function to reverse a string").
* `input`: Any provided context or starting code. (May be empty)
* `output`: The target Python code solution.

**2. The ChatML Mapping**
Our formatting function merges these columns and wraps them using Qwen's specific control tokens (`<|im_start|>` and `<|im_end|>`). The resulting string looks like this:

```text
<|im_start|>system
You are a helpful Python coding assistant.
<|im_end|>

<|im_start|>user
[instruction text] + [input text]
<|im_end|>

<|im_start|>assistant
[output Python code]
<|im_end|>
```

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.model_max_length = 1024

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# format the dataset into the standard "messages" format
def format_qwen_prompts(example):
    # input + instruction or just instruction if no input
    user_prompt = example['instruction'] if not example['input'] else f"{example['instruction']}\n\n{example['input']}"
    
    return {
        "messages": [
            {"role": "system", "content": "You are a helpful Python coding assistant."},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": example['output']}
        ]
    }

# apply mapping and remove old columns
formatted_dataset = dataset.map(format_qwen_prompts, remove_columns=dataset.column_names)

print(formatted_dataset[0]['messages'])

[{'role': 'system', 'content': 'You are a helpful Python coding assistant.'}, {'role': 'user', 'content': 'Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order'}, {'role': 'assistant', 'content': '```python\nimport random\n\n# generating a list of unique numbers from 0 to 9 in random order\nrandom_numbers = random.sample(range(0, 10), 10)\n\n# sort list of numbers \nrandom_numbers.sort()\n\n# print sorted list of random numbers\nprint(random_numbers)\n# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]\n```'}]


### Step 3: Model Quantization and LoRA Injection

This section prepares the 1.5B parameter Qwen model to train on a standard 16GB GPU by heavily reducing its memory footprint. It relies on two main configurations:

1. **4-bit Quantization (`BitsAndBytesConfig`):** Compresses the foundational model weights down to 4-bit precision. This shrinks the base model enough to fit entirely inside the VRAM.
2. **Low-Rank Adaptation (`LoraConfig`):** Freezes all 1.5 billion parameters of the base model so they require no gradient calculations. It then injects small, empty weight matrices ($A$ and $B$) into the model's linear layers. 

During training, the input ($X$) passes through both the frozen base matrix ($W$) and the newly injected matrices, summing the results to produce the final output ($Y$):$$Y = WX + ABX$$

The optimizer will only compute gradients and train the newly injected $A$ and $B$ adapters (~18.4 million parameters), making the fine-tuning process computationally possible on consumer hardware.

In [22]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

# apply LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


### Step 4: Supervised Fine-Tuning (SFT) 

- This section configures the training loop using the exact parameters specified. We use a batch size of 1 with 4 gradient accumulation steps to simulate a larger batch size while staying strictly within the 16GB VRAM limit. 
- The `paged_adamw_8bit` optimizer is utilized to page memory to the CPU if the GPU approaches its limit, preventing crashes. 
- Finally, the `report_to="wandb"` flag ensures the $\mathcal{L}_{SFT}$ metrics are logged to your Weights & Biases dashboard for your final analysis.

In [23]:
from trl import SFTTrainer
from transformers import TrainingArguments

# specified configuration 
sft_config = TrainingArguments(
    output_dir="./qwen-sft-model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=10,
    report_to="wandb",
    run_name="qwen-sft-run",
)

# initialize the trainer with PEFT-wrapped model and formatted data
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=sft_config,
)

# fine-tuning
trainer.train()

# save
trainer.save_model("./qwen-sft-model")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


### Step 5: Exporting the LoRA Adapters

After the SFT phase completes, we execute `trainer.save_model()`. It only exports a lightweight, highly portable folder containing only the newly acquired capabilities:

1. **`adapter_model.safetensors`**: The physical weight file containing the 18.4 million newly trained parameters (the independent $A$ and $B$ matrices injected across the network's linear layers).
2. **`adapter_config.json`**: The blueprint mapping out exactly how these 18.4 million parameters must be reattached to the 1.5B base model for future use (Rank 16, all-linear targets).
3. **Tokenizer Configuration**: The dictionary files (`tokenizer.json`, `vocab.json`, etc.) that preserve the specific ChatML role formatting (`<|im_start|>`) used during this training session.

This exported folder is required to initialize the upcoming Direct Preference Optimization (DPO) alignment phase.

In [30]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

source_dir = "./qwen-sft-model"
destination_dir = "/content/drive/MyDrive/qwen-sft-model-backup"

shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

print("Saved to Google Drive.")

Mounted at /content/drive
Saved to Google Drive.
